# 03 — Land-Use Efficiency: Yield vs. Area Change

**Research Question 2:** *Which EU countries achieve higher yields without expanding agricultural land?*

This notebook compares each country's yield change against its area-harvested change between two three-year periods (1995–97 and 2022–24). The result is plotted as a four-quadrant scatter, with the upper-left quadrant — *higher yield, less land* — identifying the efficiency winners.

### Methodology

- **Three-year averages** for the start (1995–97) and end (2022–24) windows smooth out single-year weather anomalies.
- **Percentage change** rather than absolute change makes countries with very different production scales (France vs. Portugal) comparable.
- **Per-country aggregation** preserves national variation that a regional aggregate would hide.


## 1. Imports and configuration


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd
import seaborn as sns

DATA_DIR = Path("../data")
FIGURES_DIR = Path("../figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")

COUNTRY_COLORS = {
    "France":      "#D62828",
    "Germany":     "#1D3557",
    "Italy":       "#1F7A6F",
    "Spain":       "#B8860B",
    "Portugal":    "#6A0572",
    "Netherlands": "#BC4B24",
}


## 2. Load processed data


In [ ]:
eu_stat_joined = pd.read_csv(DATA_DIR / "processed" / "eu_stat_joined.csv")
print(f"Shape: {eu_stat_joined.shape}")
eu_stat_joined.head()


## 3. Define the comparison windows

Three-year averaging at each endpoint reduces sensitivity to a single anomalous year (e.g. a drought or frost). The 30-year span (1995→2024) was chosen as the longest fully-covered period in the FAOSTAT extract.


In [ ]:
START_YEARS = [1995, 1996, 1997]
END_YEARS = [2022, 2023, 2024]

GROUP_COLS = ["Area", "Item"]
VALUE_COLS = ["area_ha", "Annual Yield (t/ha)"]


## 4. Compute mean values for each window

Group by country and crop, then take the mean across the three years in each window. Because both metrics live in the same dataframe and share the same index after groupby, they can be computed in a single pass per window.


In [ ]:
start_means = (
    eu_stat_joined[eu_stat_joined["Year"].isin(START_YEARS)]
    .groupby(GROUP_COLS)[VALUE_COLS]
    .mean()
)

end_means = (
    eu_stat_joined[eu_stat_joined["Year"].isin(END_YEARS)]
    .groupby(GROUP_COLS)[VALUE_COLS]
    .mean()
)

start_means.head()


## 5. Compute percentage change

Since `start_means` and `end_means` share the same index (country, crop), pandas aligns them automatically — no explicit merge is required. Renaming the columns clarifies what each represents in downstream code.


In [ ]:
pct_change = ((end_means - start_means) / start_means) * 100

pct_change = pct_change.rename(columns={
    "area_ha":             "area_pct_change",
    "Annual Yield (t/ha)": "yield_pct_change",
}).reset_index()

pct_change


## 6. Visualise the four-quadrant scatter

Each panel is one crop. Each point is one country, plotted with its area-change on the x-axis and yield-change on the y-axis. Four interpretive zones emerge naturally:

| Quadrant | Meaning |
|---|---|
| Upper-left | **Efficiency winners** — higher yield, less land |
| Upper-right | Growth through expansion — higher yield *and* more land |
| Lower-right | Inefficiency — less yield, more land |
| Lower-left | Decline — both shrinking |

### Outlier handling

Dutch grape cultivation expanded by roughly 800% from a tiny base. Plotting this raw value would compress every other point into an unreadable cluster. The x-axis is therefore clipped at ±150% for plotting only — the raw value is retained in the underlying table and would be referenced in any narrative discussion.

### Independent y-axes

`sharey=False` lets each crop's yield-change variation fill its panel. Sugar beet yields shifted by 10–60% while wheat moved only ±10%; sharing the axis would flatten the wheat panel.


In [ ]:
# Cap extreme x-values for plotting
plot_df = pct_change.copy()
X_CAP = 150
plot_df["area_plot"] = plot_df["area_pct_change"].clip(-X_CAP, X_CAP)

g = sns.relplot(
    data=plot_df,
    x="area_plot",
    y="yield_pct_change",
    hue="Area",
    palette=COUNTRY_COLORS,
    style="Area",
    col="Item",
    col_wrap=2,
    kind="scatter",
    s=140,
    height=5,
    aspect=1.3,
    facet_kws={"sharey": False},
)

for ax in g.axes.flat:
    # Reference lines
    ax.axhline(0, color="#333333", linewidth=1, linestyle="--")
    ax.axvline(0, color="#333333", linewidth=1, linestyle="--")

    # Quadrant shading on the upper-left (efficiency winners)
    xlim, ylim = ax.get_xlim(), ax.get_ylim()
    ax.axhspan(0, ylim[1], xmin=0, xmax=0.5, color="#2E7D32", alpha=0.05)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    # Quadrant labels
    ax.text(0.03, 0.80, "Efficiency\nwinners", transform=ax.transAxes,
            fontsize=10, fontweight="bold", color="#2E7D32", alpha=0.5,
            ha="left", va="top", style="italic")
    ax.text(0.97, 0.80, "Growth through\nexpansion", transform=ax.transAxes,
            fontsize=10, fontweight="bold", color="#B8860B", alpha=0.5,
            ha="right", va="top", style="italic")
    ax.text(0.03, 0.02, "Decline", transform=ax.transAxes,
            fontsize=10, fontweight="bold", color="#D84315", alpha=0.5,
            ha="left", va="bottom", style="italic")

    # Integer ticks
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))

g.set_axis_labels("Change in harvested area (%)", "Change in yield (%)")
g.set_titles("{col_name}", fontsize=13, fontweight="bold")

g.figure.suptitle(
    "Yield vs Area Change by Crop (1995–97 → 2022–24)",
    fontsize=16, fontweight="bold", y=1.05,
)
g.figure.text(
    0.5, 1.00,
    "Upper-left quadrant = higher yields without expanding land",
    ha="center", fontsize=10, color="gray",
)

sns.move_legend(
    g,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.10),
    ncol=6,
    title="Country",
    frameon=True,
)

g.figure.tight_layout()
g.figure.savefig(FIGURES_DIR / "yield_vs_area_change_by_crop.png",
                 dpi=400, bbox_inches="tight")
plt.show()


## Summary of findings

**Efficiency winners (upper-left quadrant — higher yields with less land):**
- **Germany** appears in this quadrant for sugar beet (+54% yield, ~−5% area) and grapes.
- **Spain** for sugar beet (+61% yield, sharply lower area) and grapes (+44%).
- **Italy** for sugar beet, grapes, and wheat — modest yield gains with reduced area.

**Growth through expansion (upper-right):**
- **Netherlands** for maize (+37% yield with +50% area) and grapes (where the area expansion is the ~800% outlier from a very small base).

**Decline (lower-left):**
- **France** and **Germany** for grapes (~−9% yield, modestly lower area).
- **France** for wheat (essentially flat yield, slightly lower area).

**Caveats:** The Netherlands grape result reflects a small absolute base, where percentage changes appear extreme. Portugal's missing post-2017 sugar beet data means its 2022–24 mean is calculated from earlier years — interpret with caution.

The pattern suggests that the EU's most efficient productivity gains come from sugar beet across both Northern and Southern producers, while grape production shows widespread structural decline in traditional powerhouses (France, Germany).
